# Data Preprocessing
Load raw CSV and parse all complex string columns into clean numeric/categorical features.

In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('Dataset-Laptops.csv', index_col='ID')
print(df.shape)
df.head()

(1303, 11)


,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
ID,,,,,,,,,,,
0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,71378.6832
1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,47895.5232
2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,30636.0000
3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,135195.3360
4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,96095.8080


In [2]:
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

Company             0
TypeName            0
Inches              0
ScreenResolution    0
Cpu                 0
Ram                 0
Memory              0
Gpu                 0
OpSys               0
Weight              0
Price               0
dtype: int64
Duplicates: 29


## Numeric Columns

In [3]:
df['ram_gb']     = df['Ram'].str.replace('GB', '').astype(int)
df['weight_kg']  = df['Weight'].str.replace('kg', '').astype(float)

## Screen Resolution

In [4]:
def parse_screen(s):
    s = str(s)
    touch = int('Touch' in s)
    ips   = int('IPS' in s)
    m = re.search(r'(\d{3,4})x(\d{3,4})', s)
    w, h = (int(m.group(1)), int(m.group(2))) if m else (np.nan, np.nan)
    return touch, ips, w, h

df[['touchscreen', 'ips_panel', 'res_w', 'res_h']] = (
    df['ScreenResolution'].apply(lambda x: pd.Series(parse_screen(x)))
)

## CPU

In [5]:
def parse_cpu(s):
    brand = 'Intel' if 'Intel' in s else ('AMD' if 'AMD' in s else 'Other')
    m     = re.search(r'(\d+\.\d+)GHz', s)
    speed = float(m.group(1)) if m else np.nan
    for tier in ['i9', 'i7', 'i5', 'i3', 'Xeon', 'Ryzen', 'Celeron', 'Pentium', 'Atom']:
        if tier in s:
            return brand, tier, speed
    if 'Core m' in s:
        return brand, 'Core_m', speed
    return brand, 'Other', speed

df[['cpu_brand', 'cpu_tier', 'cpu_ghz']] = df['Cpu'].apply(lambda x: pd.Series(parse_cpu(x)))

In [6]:
def cpu_gen(s):
    # Match model numbers like 7200U, 8550U, 7700HQ — first digit is gen
    m = re.search(r'\b(\d)(\d{3})[A-Z]', s)
    if m:
        return int(m.group(1))
    return 0  # unknown / no model number

df['cpu_gen'] = df['Cpu'].apply(cpu_gen)

## Storage

In [7]:
def parse_memory(s):
    s = re.sub(r'(\d+\.?\d*)TB', lambda m: str(int(float(m.group(1)) * 1024)) + 'GB', str(s))
    ssd = hdd = flash = 0
    for part in s.split('+'):
        m = re.search(r'(\d+)GB', part)
        if not m:
            continue
        size = int(m.group(1))
        if 'SSD' in part:
            ssd += size
        elif 'Flash' in part or 'eMMC' in part:
            flash += size
        else:
            hdd += size
    return ssd, hdd, flash

df[['ssd_gb', 'hdd_gb', 'flash_gb']] = df['Memory'].apply(lambda x: pd.Series(parse_memory(x)))

## GPU

In [8]:
def gpu_brand(s):
    if any(k in s for k in ['Nvidia', 'GeForce', 'Quadro', 'GTX', 'RTX']):
        return 'Nvidia'
    if any(k in s for k in ['AMD', 'Radeon', 'FirePro']):
        return 'AMD'
    return 'Intel'

def gpu_tier(s):
    # Integrated graphics
    if any(k in s for k in ['HD Graphics', 'UHD Graphics', 'Iris', 'HD 4', 'HD 5', 'HD 6']):
        return 0
    # High-end discrete
    if any(k in s for k in ['GTX 1070', 'GTX 1080', 'RTX', 'Quadro', 'FirePro', 'RX Vega']):
        return 3
    # Mid-range discrete
    if any(k in s for k in ['GTX 1050', 'GTX 1060', 'MX150', 'MX130', 'GTX 960M', 'GTX 965M',
                             'Radeon R9', 'RX 4', 'RX 5']):
        return 2
    # Budget discrete (any remaining Nvidia/AMD)
    if any(k in s for k in ['Nvidia', 'GeForce', 'Radeon', 'AMD', 'GTX', 'RTX']):
        return 1
    return 0

df['gpu_brand'] = df['Gpu'].apply(gpu_brand)
df['gpu_tier']  = df['Gpu'].apply(gpu_tier)

## OS

In [9]:
def clean_os(s):
    s = str(s)
    if 'Windows 10' in s:  return 'Win10'
    if 'Windows' in s:     return 'WinOther'
    if 'mac' in s.lower(): return 'macOS'
    if any(k in s for k in ['Linux', 'Ubuntu', 'Chrome']): return 'Linux'
    return 'NoOS'

df['os'] = df['OpSys'].apply(clean_os)

## Save

In [10]:
drop_raw = ['Ram', 'Weight', 'ScreenResolution', 'Cpu', 'Memory', 'Gpu', 'OpSys']
df_clean = df.drop(columns=drop_raw)
df_clean.to_csv('data_preprocessed.csv')
print(df_clean.shape)
df_clean.dtypes

(1303, 20)


Company            str
TypeName           str
Inches         float64
Price          float64
ram_gb           int64
weight_kg      float64
touchscreen      int64
ips_panel        int64
res_w            int64
res_h            int64
cpu_brand          str
cpu_tier           str
cpu_ghz        float64
cpu_gen          int64
ssd_gb           int64
hdd_gb           int64
flash_gb         int64
gpu_brand          str
gpu_tier         int64
os                 str
dtype: object